# 리포트 04 — 필드 갱신 인자 여덟 개에 면적·곡률·치수·λ 가 없다

> ### 한 일
> **필드 갱신 함수의 인자를 전수로 세고, 호출부가 무엇을 일부러 요청하지 않는지까지 소스 줄 번호로 적었다.**

### 결과
1. 인자는 8개이고 그중 여섯은 방향(단위벡터·회전), 둘은 Fresnel 계수다.
2. 호출부가 `return_vertices=False` 로 **정점 좌표를 일부러 요청하지 않고** 법선만 가져온다(`field_calculator.py:404-405`) — 삼각형 크기를 알 수 있는 유일한 통로가 그 자리에서 닫힌다.
3. 정확한 진술은 이것이다 — 광선이 면을 맞았는가는 유한 기하로 판정하고, 맞은 뒤 필드를 얼마나 바꿀지는 국소 평면파–평면경계 문제의 해로 계산한다. 즉 크기는 `yes/no` 에만 쓰인다.
4. 단위가 이미 답을 말한다. 같은 PEC, 같은 정면면적 0.7854 [^1] m², 같은 5G 밴드에서 구는 -1.05 dBsm [^2], 평판은 30.24 dBsm [^3] 로 31.29 dB [^4] 갈린다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 인자 전수 | 설치본 소스의 시그니처를 그대로 옮기고 `inspect.signature` 로 런타임 재확인했다 |
| 원문 인용 | 기술보고서 원문 문장을 축자로 싣는다 — «무한평면 가정» 같은 거친 요약을 쓰지 않기 위해서다 |
| 모양 대조 | 같은 재질·같은 정면면적에서 구와 평판의 σ 를 닫힌형으로 계산한다 |

### 재현

```bash
PYTHONPATH=src python src/figs_report00.py
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_sionna_anatomy.json`, `outputs/report00_evidence.json`, `outputs/report00_po_case.json` |
| 소요 | 약 1분 (GPU 0장) |

---

## 그 수식에 없는 것 — 인자 여덟 개를 그대로 센다

필드 갱신 함수가 받는 것을 전수로 옮긴다. 세어 보면 방향과 계수뿐이다.

| 인자 | 무엇인가 |
|---|---|
| to_world | 국소→월드 3x3 회전 행렬 (자세, 크기 아님) |
| ki_local | 입사 진행 방향 단위벡터 |
| ko_local | 산란 진행 방향 단위벡터 |
| reflection | 반사인가 투과인가 하는 불리언 |
| r_te | TE 반사계수 (Fresnel, 복소) |
| r_tm | TM 반사계수 (Fresnel, 복소) |
| t_te | TE 투과계수 (복소) |
| t_tm | TM 투과계수 (복소) |

출처 [^5]

## 목록 밖에 있는 양들

| 필드 갱신 인자 목록 밖에 있는 양 |
|---|
| 표면 곡률 (주곡률반경 R1, R2) |
| 물체 면적 A |
| 물체 치수 L (전장·폭·높이) |
| 부딪힌 삼각형의 면적 또는 변 길이 |
| 삼각형 정점 좌표 (return_vertices=False 로 명시적으로 요청하지 않음) |
| 인접 삼각형 정보 (회절을 끈 경우) |
| 파장 (이 함수 안에는 없음; 상위 Fresnel 계수 계산에만 들어감) |

출처 [^6]

여덟 인자 중 여섯은 방향(단위벡터·회전)이고 둘은 Fresnel 계수다. 더 결정적인 것은 호출부다 — `field_calculator.py:404-405` 가 `return_vertices=False` 로 **정점 좌표를 일부러 요청하지 않고** 법선만 가져온다. 삼각형 크기를 알 수 있는 유일한 통로가 그 자리에서 닫힌다.

## 정확히 말하는 법 — «무한평면 가정» 은 거친 요약이다

Sionna 가 무한평면을 가정한다고 쓰면 반박당한다. 기하는 유한하고, 광선이 그 삼각형을 맞았는지는 Mitsuba 가 정확히 판정한다 — 가림·그림자는 제대로 작동한다.

정확한 진술은 이것이다: **광선이 면을 맞았는가는 유한 기하로 판정하고, 맞은 뒤 필드를 얼마나 바꿀지는 국소 평면파–평면경계 문제의 해로 계산한다.** 즉 크기는 `yes/no` 에만 쓰이고, `how much` 는 국소 해가 정한다.

· 경로 기하 — p.19 — "the image method assumes all reflection surfaces extend infinitely, making the exact in-plane position of a primitive irrelevant to the path geometry" [^7]

· 계수 — p.46 — "The reflection and refraction coefficients described above assume that the object reflecting the wave or allowing it to penetrate is of infinite size (or thickness)." [^8]

· ⚠ 낱말 주의 — 기술보고서의 «locally planar» 는 **파(wave)** 에 붙는 말이다(p.50 “an incoming locally planar linearly polarized wave”). 표면에 대해서는 조건 없이 extend infinitely · of infinite size 라고 쓴다 (근거 `outputs/report00_evidence.json : G_exact_wording_infinite_surface.numbers.caution_locally`).

## 단위가 이미 답을 말한다 — Γ 는 무차원, σ 는 m²

반사계수 Γ 는 무차원이고 레이더단면적 σ 의 단위는 m² 다. 무차원을 아무리 정확히 계산해도 결과는 무차원으로 남는다. 면적은 **조명면 위의 면적분**에서만 들어온다.

평판의 PO 공식 `σ = 4πA²/λ²` 는 `4π·[m²]²/[m]² = m²` 로 닫히는데, Sionna 의 정반사 진폭 `|a| = λ/(4π(R₁+R₂))` 는 `[m]/[m] = 1` 로 닫힌다 — 면적 기호는 그 식 밖에 있다.

## 같은 면적, 다른 모양 — σ 는 얼마나 갈리는가

![report00_f3](../outputs/figures/report00_f3.png)

**그림 1.** 같은 재질·같은 정면면적에서 모양만 바꾸면 σ 는 얼마나 갈라지는가?

같은 PEC, 같은 정면면적 0.7854 [^1] m², 같은 5G 밴드 3.5 GHz [^9] 에서 구는 -1.05 dBsm [^2], 평판은 30.24 dBsm [^3] 다.

주파수를 두 배로 올리면 평판은 +6.02 dB [^10]/옥타브, 구는 +0.00 dB [^11]/옥타브 움직인다. 갈라지는 이유는 하나다 — 두 값 모두 |Γ|=1 을 쓴다. 차이는 위상이 면 위에서 어떻게 정렬되는가다 — 평판은 A 전체가 같은 위상으로 더해지고(∝A), 구는 곡률이 위상을 흩어 실효 기여면이 λ 규모의 정반사점 근방으로 줄어든다. [^12]

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 표적 크기를 흔들어 경로 진폭이 움직이는지 실행으로 잰다 | «크기는 yes/no 에만 쓰인다» 가 dB 로 확정된다 | [편 05 «면적을 1600배로 키워도 경로 진폭은 7.4…»](05_size-sweep.ipynb) |
| 면적분을 얹어 σ 를 m² 로 만드는 커널의 분담선을 적는다 | 가림 판정과 면적분의 경계가 코드 경계로 확정된다 | [편 18 «가림 판정은 Sionna 광선엔진이 하고»](18_kernel-what.ipynb) |
| 어느 실험이 이 경계를 실제로 넘어야 하는지 결정표로 가른다 | 표적 모델이 필요한 실험과 필요 없는 실험이 칸으로 확정된다 | [편 06 «표적 항이 비에서 소거되는가와 절대값이 필요한가»](06_decision-table.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 12개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.frontal_area_m2` | 0.7854 |
| [^2] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.sphere_sigma_dbsm` | -1.049 |
| [^3] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.plate_same_area_sigma_dbsm` | 30.24 |
| [^4] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.shape_gap_db` | 31.29 |
| [^5] | `outputs/report00_sionna_anatomy.json` | `item2_field_calculation_arguments.argument_inventory` | (8항목 묶음) |
| [^6] | `outputs/report00_sionna_anatomy.json` | `item2_field_calculation_arguments.absent_quantities` | (7행 표) |
| [^7] | `outputs/report00_evidence.json` | `G_exact_wording_infinite_surface.numbers.quote_path_geometry` | p.19 — "the image method assumes all reflection surface… |
| [^8] | `outputs/report00_evidence.json` | `G_exact_wording_infinite_surface.numbers.quote_coefficients` | p.46 — "The reflection and refraction coefficients desc… |
| [^9] | `outputs/report00_po_case.json` | `s4_limits.our_production_bands_vs_knee.nr_ghz` | 3.5 |
| [^10] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.plate_sigma_df_db_per_octave` | 6.021 |
| [^11] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.sphere_sigma_df_db_per_octave` | 0 |
| [^12] | `outputs/report00_evidence.json` | `C_same_material_different_shape.formula.why_they_differ` | 두 값 모두 \|Γ\|=1 을 쓴다. 차이는 위상이 면 위에서 어떻게 정렬되는가다 — 평판은 A 전체가… |